In [2]:
from langchain_core.prompts import ChatPromptTemplate

#### ‘목표’를 표현하는 데이터 모델

In [ ]:
# '목표'의 데이터 모델은 Pydantic의 BaseModel을 이요해 표현
class Goal(BaseModel):
    description: str = Field(..., description="목표 설명")

    @property
    def text(self) -> str:
        return f"{self.description}"

#### 프롬프트와 체인

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "사용자 입력을 분석하여 명확하고 실행 가능한 목표를 생성해 주세요.\n"
    "요건:\n"
    "1. 목표는 구체적이고 명확해야 하며, 실행 가능한 수준으로 상세화되어야 합니다.\n"
    "2. 당신이 실행할 수 있는 행동은 다음과 같은 행동뿐입니다.\n"
    "    - 인터넷을 이용하여 목표 달성을 위한 조사를 수행합니다.\n"
    "    - 사용자를 위한 보고서를 생성합니다.\n"
    "3. 절대 2.에 명시된 행동 외의 다른 행동을 취해서는 안 됩니다.\n"
    "사용자 입력: {query}"
)

In [ ]:
# LLM의 출력을 Goal 모델의 구조에 맞추기 위해 with_structured_output을 사용
chain = prompt | self.llm.with_structured_output(Goal)
return chain.invoke({"query: query"})

In [ ]:
# 호출하는 측
goal_creator = PassiveGoalCreator(llm=llm)
result: Goal = goal_creator.run(query=args.task)

#### Settings 클래스에 관해

In [ ]:
import os

from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file="../.env",
        env_file_encoding="utf-8",
    )

    OPENAI_API_KEY: str
    ANTHROPIC_API_KEY: str = ""
    TAVILY_API_KEY: str
    LANGSMITH_ENDPOINT: str = "https://aws.api.smith.langchain.com"
    LANGSMITH_AGENT_ID: str = "llm-engineer-roadmap"
    LANGSMITH_AGENT_ENVIRONMENT: str = "local"

    openai_smart_model: str = "gpt-6-luna"
    embedding_model: str = "intflat/multilingual-e5-base"
    anthropic_smart_model: str = ""
    temperature: float = 0.0
    default_reflection_db_path: str = "tmp/reflection_db.json"

    def __init__(self, **values):
        super().__init__(**values)
        self._set_env_variables()
    
    def _set_env_variables(self):
        for key in self.__annotations__.keys():
            if key.isupper():
                os.environ[key] = getattr(self, key)